#### Stored Procedure 

What is a Stored Procedure?

A stored procedure is a prepared SQL code that you can save, so the code can be reused over and over again.

So if you have an SQL query that you write over and over again, save it as a stored procedure, and then just call it to execute it.

You can also pass parameters to a stored procedure, so that the stored procedure can act based on the parameter value(s) that is passed.

In [8]:
import mysql.connector

conn = mysql.connector.connect(  #establishing the connection
    host = "localhost",
    user="root",
    passwd = "mYsql@2022",
    database = "saleorder"
)

In [9]:
conn

In [10]:
c = conn.cursor()

In [11]:
c

SQL procedures are a set of SQL statements grouped together to form a logical unit of work. They are similar to functions or methods in programming languages, enabling you to encapsulate complex queries and operations into a single reusable entity.

Procedures enhance code modularity, readability, and maintainability, making it easier to manage and execute repetitive or intricate database tasks.

In [ ]:
c.execute("""
    DELIMITER $$

    CREATE PROCEDURE GenerateSalesReport (
        IN start_date DATE,
        IN end_date DATE
    )
    BEGIN
        SELECT 
            DATE_FORMAT(SaleDate, '%Y-%m-%d') AS `Date`,
            SaleQuantity AS TotalOrders,
            SUM(SaleQuantity * SaleUnitPrice) AS TotalSales
        FROM sale
        WHERE SaleDate BETWEEN start_date AND end_date
        GROUP BY DATE_FORMAT(SaleDate, '%Y-%m-%d'), SaleQuantity;
    END $$

    DELIMITER ;
""")

In [13]:
c.callproc('GenerateSalesReport', ['2022-01-01', '2022-02-01'])

('2022-01-01', '2022-02-01')

In [14]:
for result in c.stored_results():
    print(result.fetchall())

[('2022-01-01', 1, Decimal('2000.00')), ('2022-01-05', 1, Decimal('500.00')), ('2022-01-10', 1, Decimal('2250.00')), ('2022-01-15', 1, Decimal('1000.00')), ('2022-01-20', 1, Decimal('4000.00')), ('2022-01-25', 1, Decimal('1500.00')), ('2022-02-01', 1, Decimal('3600.00'))]


In [18]:
c.execute("""
        DELIMITER $$
        
        CREATE PROCEDURE Temp_Employee(
            IN JobTitle varchar(100) 
        ) 
        BEGIN 
        
            DROP TEMPORARY TABLE IF EXISTS temp_employee; 
            CREATE TEMPORARY TABLE temp_employee (
            JobTitle VARCHAR(100),
            EmployeesPerJob INT,
            AvgAge INT,
            AvgSalary INT
            ); 
            
            INSERT INTO temp_employee (JobTitle, EmployeesPerJob, AvgAge, AvgSalary)
            SELECT 
                emp.JobTitle,
                COUNT(emp.EmployeeID) AS EmployeesPerJob,
                CAST(AVG(emp.Age) AS SIGNED) AS AvgAge,
                CAST(AVG(sal.salary) AS SIGNED) AS AvgSalary
            FROM employeedemo emp
            JOIN Salary sal ON emp.EmployeeID = sal.EmployeeID
            WHERE emp.JobTitle = JobTitle
            GROUP BY emp.JobTitle; 
            
            Select *  From temp_employee ;
        
        END $$
        
        DELIMITER ;

        """)




In [20]:
c.callproc('Temp_Employee',['Software Engineer'])

('Software Engineer',)

In [21]:
for result in c.stored_results():
    print(result.fetchall())

[('Software Engineer', 1, 30, 50000)]
